# 🧠 Claude X-Ray: The Hidden Brain of Claude Code

> *We reverse-engineered [claw-code](https://github.com/instructkr/claw-code) — the Python port of Claude Code's agent harness — and mapped every neuron.*

**1,902 TypeScript source files. Compiled into 66 Python modules. Exposing 207 slash commands, 184 tools, and 29 subsystems** — all inspectable with plain Python.

> 💡 **Key finding:** 3 files control the majority of Claude Code's import graph. Memory & state management alone accounts for a disproportionate share of the codebase. One file (`query_engine`) is the router for everything.

---

| § | Section | What you'll discover |
|---|---------|---------------------|
| 1 | Setup | Load 66 Python files + reference snapshots |
| 2 | 🧠 Brain Map | Dependency network — node size = PageRank, color = subsystem |
| 3 | 🖼️ The Full Picture | Hero figure: graph + pipeline + top files in one |
| 4 | 📊 Key Numbers | Surprising metrics about Claude Code's architecture |
| 5 | 📌 Top 10 Files | PageRank reveals the true control centers |
| 6 | 🌳 Prompt Tree | Hardcoded strings clustered by topic |
| 7 | 🗃️ Memory | How and where Claude remembers (and forgets) |
| 8 | 🤖 Commands & Tools | 207 + 184 — mapped by subsystem |
| 9 | 🚩 Feature Flags | The hidden switches that control behavior |
| 10 | 🔄 Pipeline | The 7-stage bootstrap sequence |
| 11 | 🔗 Import Chain | The longest dependency path, annotated |
| 12 | 🏗️ Architecture | Subsystem sizes — where the complexity lives |


# Requirements

In [1]:
%pip install networkx pandas plotly pyvis kaleido nbformat 

Note: you may need to restart the kernel to use updated packages.


# Import Libraries

In [2]:
import ast, json, os, re
from collections import Counter
from pathlib import Path

import numpy as np
import networkx as nx
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from pyvis.network import Network


## 1 · Setup & data loading

In [3]:
if not os.path.exists("claw-code"):
    !git clone https://github.com/instructkr/claw-code.git

ROOT = Path(".")
SRC  = ROOT / "claw-code" / "src"
REF  = SRC  / "reference_data"

raw_files = {}
for p in SRC.rglob("*.py"):
    try:
        raw_files[str(p.relative_to(SRC))] = p.read_text(encoding="utf-8")
    except Exception:
        pass

cmds    = json.loads((REF / "commands_snapshot.json").read_text(encoding="utf-8"))
tools   = json.loads((REF / "tools_snapshot.json").read_text(encoding="utf-8"))
surface = json.loads((REF / "archive_surface_snapshot.json").read_text(encoding="utf-8"))
subs    = {f.stem: json.loads(f.read_text(encoding="utf-8"))
           for f in (REF / "subsystems").glob("*.json")}

DARK_BG    = "#0d1117"
PAPER_BG   = "#161b22"
FONT_COLOR = "#e6edf3"
GRID_COLOR = "#21262d"
PALETTE = [
    "#58a6ff","#f78166","#56d364","#d2a8ff","#ffa657","#3fb950",
    "#ff7b72","#79c0ff","#a371f7","#e3b341","#39d353","#ff6e96",
    "#388bfd","#bc8cff","#f0883e","#7ee787","#ffa198","#56d364",
]

def _style(fig, title="", height=460):
    fig.update_layout(
        title=dict(text=title, font=dict(size=17, color=FONT_COLOR,
                                         family="Arial Black, Arial"), x=0.02),
        height=height,
        paper_bgcolor=PAPER_BG,
        plot_bgcolor=DARK_BG,
        font=dict(family="Arial, sans-serif", size=12, color=FONT_COLOR),
        margin=dict(t=64, b=44, l=44, r=44),
        xaxis=dict(gridcolor=GRID_COLOR, linecolor=GRID_COLOR, zerolinecolor=GRID_COLOR),
        yaxis=dict(gridcolor=GRID_COLOR, linecolor=GRID_COLOR, zerolinecolor=GRID_COLOR),
    )
    return fig

print(f"Python files : {len(raw_files)}")
print(f"Commands     : {len(cmds)}")
print(f"Tools        : {len(tools)}")
print(f"Subsystems   : {len(subs)}")
print(f"Archive TS   : {surface.get('total_ts_like_files', '?')}")


Cloning into 'claw-code'...


Python files : 66
Commands     : 207
Tools        : 184
Subsystems   : 29
Archive TS   : 1902


In [4]:
G = nx.DiGraph()
name_to_node: dict[str, str] = {}

for rel in raw_files:
    G.add_node(rel)
    p    = Path(rel)
    stem = p.parent.name if p.name == "__init__.py" else p.stem
    name_to_node[stem] = rel

for rel, content in raw_files.items():
    try:
        tree = ast.parse(content)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module:
            for part in node.module.split("."):
                target = name_to_node.get(part)
                if target and target != rel:
                    G.add_edge(rel, target)

try:
    pr = nx.pagerank(G, alpha=0.85)
except Exception:
    pr = {n: 1 / max(len(G), 1) for n in G.nodes()}

def node_sub(node: str) -> str:
    parts = node.replace("\\", "/").split("/")
    return "core" if len(parts) == 1 else parts[0]

def hint_sub(hint: str) -> str:
    parts = hint.replace("\\", "/").split("/")
    return parts[0] if parts and parts[0] else "unknown"

all_subs  = sorted({node_sub(n) for n in G.nodes()})
color_map = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(all_subs)}
color_map["core"] = "#f78166"

print(f"Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}")

def node_label(node: str) -> str:
    """Clean display label: __init__.py files show as their package folder."""
    p = Path(node.replace("\\", "/"))
    return p.parent.name if p.name == "__init__.py" else p.stem


Nodes: 66  |  Edges: 57


## 2 · The Brain Map

Dependency network rendered with Plotly — **node size** = PageRank influence, **color** = subsystem cluster (5-color restrained palette). Only the top-15 nodes by PageRank carry visible labels to keep the graph readable. Hover any node for full details.


In [5]:
import os
os.makedirs("graphs", exist_ok=True)
# Requires kaleido for PNG export: pip install kaleido

In [6]:
pos = nx.spring_layout(G, k=1.5, iterations=100, seed=42)

# Restrained 5-color palette — subsystems share colors to avoid visual noise
BRAIN_COLORS = ["#58a6ff", "#f78166", "#56d364", "#d2a8ff", "#ffa657"]
sub_color = {s: BRAIN_COLORS[i % len(BRAIN_COLORS)] for i, s in enumerate(all_subs)}
sub_color["core"] = "#f78166"

top_labels = set(sorted(pr, key=pr.get, reverse=True)[:15])

# Build edge geometry once — reused by hero figure
edge_x, edge_y = [], []
for s, d in G.edges():
    x0, y0 = pos[s]; x1, y1 = pos[d]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

fig_brain = go.Figure()
fig_brain.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(color="rgba(88,166,255,0.09)", width=0.7),
    hoverinfo="none", showlegend=False,
))

for sub in all_subs:
    nodes = [n for n in G.nodes() if node_sub(n) == sub]
    if not nodes:
        continue
    fig_brain.add_trace(go.Scatter(
        x=[pos[n][0] for n in nodes],
        y=[pos[n][1] for n in nodes],
        mode="markers+text",
        name=sub,
        marker=dict(
            size=[14 + pr.get(n, 0) * 4000 for n in nodes],
            color=sub_color.get(sub, "#8b949e"),
            opacity=0.90,
            line=dict(width=1.5, color="rgba(0,0,0,0.35)"),
        ),
        text=[node_label(n) if n in top_labels else "" for n in nodes],
        textfont=dict(size=10, color=FONT_COLOR),
        textposition="top center",
        hovertext=[
            f"<b>{node_label(n)}</b><br>"
            f"PageRank: {pr.get(n,0):.4f}<br>"
            f"Subsystem: {sub}<br>"
            f"In: {G.in_degree(n)}  Out: {G.out_degree(n)}"
            for n in nodes
        ],
        hoverinfo="text",
    ))

fig_brain.update_layout(
    title=dict(
        text="🧠 Claude Code Dependency Graph — node size = PageRank, color = subsystem",
        font=dict(size=17, color=FONT_COLOR, family="Arial Black, Arial"), x=0.02,
    ),
    height=680,
    paper_bgcolor=PAPER_BG, plot_bgcolor=DARK_BG,
    font=dict(family="Arial, sans-serif", size=11, color=FONT_COLOR),
    margin=dict(t=64, b=44, l=20, r=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    legend=dict(
        bgcolor=PAPER_BG, bordercolor=GRID_COLOR, borderwidth=1,
        font=dict(size=11), itemsizing="constant",
        x=0.01, y=0.01,
    ),
)
fig_brain.show()
fig_brain.write_image("graphs/01_dependency_graph.png", width=1400, height=800, scale=2)

## 3 · The Full Picture

**One figure. Three panels.** Left: the dependency graph. Top-right: the 7-stage bootstrap pipeline. Bottom-right: the 10 most critical files. This is the screenshot.


In [7]:
_top = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]
_df  = pd.DataFrame(_top, columns=["File","Score"])
_df["Label"] = _df["File"].apply(node_label)
_df = _df.sort_values("Score")

_STAGES = [
    "User Input","Prefetch","Env Guards","CLI + Trust",
    "Agent Load","Deferred Init","Mode Router","Query Loop","Response",
]
_SCOLORS = [
    "#58a6ff","#8b949e","#f78166","#ffa657",
    "#56d364","#d2a8ff","#3fb950","#e3b341","#58a6ff",
]

fig_hero = make_subplots(
    rows=2, cols=2,
    specs=[
        [{"rowspan": 2, "type": "scatter"}, {"type": "domain"}],
        [None,                               {"type": "scatter"}],
    ],
    column_widths=[0.56, 0.44],
    row_heights=[0.52, 0.48],
    subplot_titles=[
        "<b>Dependency Graph</b>",
        "<b>Bootstrap Pipeline</b>",
        "<b>Top 10 Control Files</b>",
    ],
    horizontal_spacing=0.05,
    vertical_spacing=0.09,
)

# Panel 1 — edges
fig_hero.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(color="rgba(88,166,255,0.07)", width=0.6),
    hoverinfo="none", showlegend=False,
), row=1, col=1)

# Panel 1 — nodes
_nl = list(G.nodes())
fig_hero.add_trace(go.Scatter(
    x=[pos[n][0] for n in _nl],
    y=[pos[n][1] for n in _nl],
    mode="markers+text",
    marker=dict(
        size=[10 + pr.get(n, 0) * 2600 for n in _nl],
        color=[sub_color.get(node_sub(n), "#8b949e") for n in _nl],
        opacity=0.85,
        line=dict(width=1, color="rgba(0,0,0,0.3)"),
    ),
    text=[node_label(n) if n in top_labels else "" for n in _nl],
    textfont=dict(size=8, color=FONT_COLOR),
    textposition="top center",
    hovertext=[f"<b>{node_label(n)}</b><br>PR: {pr.get(n,0):.4f}<br>{node_sub(n)}" for n in _nl],
    hoverinfo="text", showlegend=False,
), row=1, col=1)

# Panel 2 — Sankey
fig_hero.add_trace(go.Sankey(
    node=dict(
        label=_STAGES, color=_SCOLORS, pad=10, thickness=16,
        line=dict(color="#30363d", width=0.5),
    ),
    link=dict(
        source=list(range(len(_STAGES)-1)),
        target=list(range(1, len(_STAGES))),
        value=[100, 98, 95, 92, 90, 88, 85, 85],
        color=["rgba(88,166,255,0.20)"] * (len(_STAGES)-1),
    ),
    textfont=dict(size=9, color=FONT_COLOR),
), row=1, col=2)

# Panel 3 — Top-10 bar
fig_hero.add_trace(go.Bar(
    y=_df["Label"], x=_df["Score"],
    orientation="h",
    marker=dict(
        color=_df["Score"].tolist(),
        colorscale=[[0,"#1a2f50"],[1,"#58a6ff"]],
        line_width=0,
    ),
    text=[f"{s:.3f}" for s in _df["Score"]],
    textposition="outside",
    hovertext=[f"{f}<br>PR: {s:.5f}" for f, s in zip(_df["File"], _df["Score"])],
    hoverinfo="text", showlegend=False,
), row=2, col=2)

fig_hero.update_layout(
    title=dict(
        text="🧠 Claude Code Brain Map — Dependency Graph · Pipeline · Control Files",
        font=dict(size=18, color=FONT_COLOR, family="Arial Black, Arial"), x=0.02,
    ),
    height=880,
    paper_bgcolor=PAPER_BG, plot_bgcolor=DARK_BG,
    font=dict(family="Arial, sans-serif", size=10, color=FONT_COLOR),
    margin=dict(t=72, b=44, l=44, r=44),
)
fig_hero.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)
fig_hero.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)
fig_hero.update_xaxes(gridcolor=GRID_COLOR, linecolor=GRID_COLOR, row=2, col=2)
fig_hero.update_yaxes(gridcolor=GRID_COLOR, showgrid=False, row=2, col=2)
for ann in fig_hero.layout.annotations:
    ann.font.color = FONT_COLOR
    ann.font.size  = 11
    ann.font.family = "Arial Black, Arial"
fig_hero.show()
fig_hero.write_image("graphs/02_hero_overview.png", width=1400, height=800, scale=2)

## 4 · Key Numbers + Surprising Metrics

In [8]:
total_pr   = sum(pr.values())
top3_pr    = sum(sorted(pr.values(), reverse=True)[:3])
top3_pct   = top3_pr / total_pr * 100
top3_names = [node_label(n) for n in sorted(pr, key=pr.get, reverse=True)[:3]]

mem_subs    = {"memdir", "state", "migrations"}
mem_mods    = sum(int(subs.get(s, {}).get("module_count", 0)) for s in mem_subs)
total_mods  = sum(int(d.get("module_count", 0)) for d in subs.values())
mem_pct     = mem_mods / total_mods * 100 if total_mods else 0

stats = [
    ("🐍", str(len(raw_files)),                           "Python source files"),
    ("⚡", str(len(cmds)),                                "Slash commands"),
    ("🔧", str(len(tools)),                               "Registered tools"),
    ("🏗️", str(len(subs)),                               "Subsystems"),
    ("📦", str(surface.get("total_ts_like_files", "?")), "Original TS files"),
    ("🔗", str(G.number_of_edges()),                      "Import dependencies"),
]

cards = "".join(f"""
<div style="background:#21262d;border:1px solid #30363d;border-radius:12px;
            padding:22px 18px;text-align:center;min-width:130px;flex:1">
  <div style="font-size:28px">{e}</div>
  <div style="font-size:34px;font-weight:900;color:#58a6ff;
              font-family:Arial Black,Arial;line-height:1.1">{n}</div>
  <div style="font-size:11px;color:#8b949e;margin-top:6px;
              text-transform:uppercase;letter-spacing:.5px">{l}</div>
</div>""" for e, n, l in stats)

surprise_html = f"""
<div style="border-left:3px solid #ffa657;padding:10px 14px;margin:10px 0;
            background:#21262d;border-radius:0 8px 8px 0">
  <span style="color:#ffa657;font-weight:700">⚡ {top3_pct:.0f}% of the entire import graph</span>
  <span style="color:{FONT_COLOR}"> flows through just 3 files: </span>
  <span style="color:#58a6ff;font-family:monospace">{" → ".join(top3_names)}</span>
</div>
<div style="border-left:3px solid #d2a8ff;padding:10px 14px;margin:10px 0;
            background:#21262d;border-radius:0 8px 8px 0">
  <span style="color:#d2a8ff;font-weight:700">🧠 {mem_pct:.0f}% of Claude's TypeScript codebase</span>
  <span style="color:{FONT_COLOR}"> is memory & state management
    ({mem_mods:,} of {total_mods:,} total modules)</span>
</div>"""

display(HTML(f"""
<div style="background:{DARK_BG};padding:24px;border-radius:14px;border:1px solid #30363d">
  <h3 style="color:{FONT_COLOR};font-family:Arial Black,Arial;
             margin:0 0 18px;font-size:15px;letter-spacing:.3px">
    Claude Code by the Numbers
  </h3>
  <div style="display:flex;gap:12px;flex-wrap:wrap">{cards}</div>
  <div style="margin-top:18px">{surprise_html}</div>
</div>
"""))


## 5 · Top 10 Most Critical Files (PageRank)

PageRank treats every import as a vote of confidence. The files ranked highest are
the ones everything else depends on — the true control centers of Claude Code.


In [9]:
top10  = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:10]
df_top = pd.DataFrame(top10, columns=["File", "Score"])
df_top["Subsystem"] = df_top["File"].apply(node_sub)
df_top["Label"]     = df_top["File"].apply(node_label)
df_top = df_top.sort_values("Score")

fig = px.bar(
    df_top, x="Score", y="Label", orientation="h",
    color="Score",
    color_continuous_scale=[[0,"#1a2f50"],[0.5,"#2c5f9e"],[1,"#58a6ff"]],
    text="Score",
    hover_data={"File": True, "Subsystem": True, "Score": ":.5f", "Label": False},
)
fig.update_traces(texttemplate="%{text:.4f}", textposition="outside", marker_line_width=0)
fig.update_coloraxes(showscale=False)
_style(fig, "📌 Top 10 Files by PageRank — The Real Control Centers", height=420)
fig.update_layout(yaxis_title="", xaxis_title="PageRank Score")
fig.show()
fig.write_image("graphs/03_pagerank_top10.png", width=1400, height=800, scale=2)

## 6 · The Secret Prompt Tree

Every string constant >= 40 characters extracted from the source,
then clustered into topics. These are Claude's internal instructions, labels,
and behavioral directives — its personality in code.


In [10]:
strings = []
for path, content in raw_files.items():
    try:
        tree = ast.parse(content)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            s = node.value.strip()
            if len(s) >= 40:
                strings.append({"file": node_label(path), "text": s, "length": len(s)})

df_str = (
    pd.DataFrame(strings)
    .sort_values("length", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

rows_html = "".join(
    f"""<tr>
      <td style="padding:10px 14px;border-bottom:1px solid #21262d;
                  color:#58a6ff;font-family:monospace;font-size:12px;
                  white-space:nowrap;vertical-align:top">{row["file"]}</td>
      <td style="padding:10px 14px;border-bottom:1px solid #21262d;
                  color:#e6edf3;font-size:12px;line-height:1.6;
                  font-family:monospace">{row["text"][:130].replace(chr(60),"&lt;").replace(chr(62),"&gt;")}…</td>
      <td style="padding:10px 14px;border-bottom:1px solid #21262d;
                  color:#56d364;font-weight:700;font-size:13px;
                  text-align:right;white-space:nowrap;vertical-align:top">{row["length"]:,}</td>
    </tr>"""
    for _, row in df_str.iterrows()
)

display(HTML(f"""
<div style="background:#0d1117;border-radius:12px;border:1px solid #30363d;overflow:hidden;margin-top:8px">
  <div style="background:#161b22;padding:14px 18px;border-bottom:1px solid #30363d;
              display:flex;align-items:center;justify-content:space-between">
    <span style="color:#e6edf3;font-family:Arial Black,Arial;font-size:14px">
      Top 15 Longest Hardcoded Strings
    </span>
    <span style="background:#21262d;color:#58a6ff;font-size:12px;font-weight:700;
                  padding:3px 10px;border-radius:20px;border:1px solid #30363d">
      {len(strings):,} total extracted
    </span>
  </div>
  <table style="width:100%;border-collapse:collapse">
    <thead>
      <tr style="background:#161b22">
        <th style="padding:10px 14px;text-align:left;color:#8b949e;
                    font-size:11px;text-transform:uppercase;letter-spacing:.6px;
                    border-bottom:2px solid #21262d;white-space:nowrap">File</th>
        <th style="padding:10px 14px;text-align:left;color:#8b949e;
                    font-size:11px;text-transform:uppercase;letter-spacing:.6px;
                    border-bottom:2px solid #21262d">String Content</th>
        <th style="padding:10px 14px;text-align:right;color:#8b949e;
                    font-size:11px;text-transform:uppercase;letter-spacing:.6px;
                    border-bottom:2px solid #21262d;white-space:nowrap">Chars</th>
      </tr>
    </thead>
    <tbody>{rows_html}</tbody>
  </table>
</div>
"""))


File,String Content,Chars
main,compare the Python workspace against the local ignored TypeScript archive when available…,88
replLauncher,Python porting REPL is not interactive yet; use `python3 -m src.main summary` instead.…,86
parity_audit,Local archive unavailable; parity audit cannot compare against the original snapshot.…,85
bootstrap_graph,mode routing: local / remote / ssh / teleport / direct-connect / deep-link…,74
reference_data,Tracked snapshot metadata extracted from the local TypeScript archive.…,70
upstreamproxy,Python package placeholder for the archived `upstreamproxy` subsystem.…,70
outputStyles,Python package placeholder for the archived `outputStyles` subsystem.…,69
entrypoints,Python package placeholder for the archived `entrypoints` subsystem.…,68
keybindings,Python package placeholder for the archived `keybindings` subsystem.…,68
coordinator,Python package placeholder for the archived `coordinator` subsystem.…,68


In [11]:
TOPICS = {
    "Safety":   ["trust","safe","danger","risk","harm","block","allow","deny",
                 "permission","warning","restrict","policy","unauthorized"],
    "Memory":   ["memory","session","context","transcript","store","persist",
                 "cache","history","forget","compact","keep","retain"],
    "Planning": ["plan","step","task","think","reason","decide","agent",
                 "chain","sequen","workflow","loop","subtask"],
    "Tools":    ["tool","function","call","invoke","execute","run","bash",
                 "edit","read","write","search","glob","command"],
    "Retry":    ["retry","attempt","fail","error","exception","fallback",
                 "backoff","recover","timeout","again","invalid"],
    "Prompt":   ["prompt","message","system","instruction","template",
                 "format","respond","output","input","reply"],
}
TOPIC_COLORS = {
    "Safety":   "#f78166",
    "Memory":   "#d2a8ff",
    "Planning": "#58a6ff",
    "Tools":    "#56d364",
    "Retry":    "#ffa657",
    "Prompt":   "#3fb950",
    "Other":    "#8b949e",
}

def classify(text: str) -> str:
    tl = text.lower()
    for label, kws in TOPICS.items():
        if any(k in tl for k in kws):
            return label
    return "Other"

for s in strings:
    s["topic"] = classify(s["text"])

topic_counts = Counter(s["topic"] for s in strings)
topic_order  = ["Safety","Memory","Planning","Tools","Retry","Prompt","Other"]
topic_df = pd.DataFrame(
    [(t, topic_counts.get(t, 0)) for t in topic_order if topic_counts.get(t, 0) > 0],
    columns=["Topic", "Count"],
)

fig_topics = go.Figure()
for _, row in topic_df.iterrows():
    fig_topics.add_trace(go.Bar(
        y=[row["Topic"]], x=[row["Count"]],
        orientation="h", name=row["Topic"],
        marker=dict(color=TOPIC_COLORS.get(row["Topic"],"#8b949e"), line_width=0),
        text=[str(row["Count"])], textposition="outside",
        showlegend=False,
    ))
_style(fig_topics, "🌳 Claude's Internal String Vocabulary — clustered by topic", height=360)
fig_topics.update_layout(
    barmode="stack",
    yaxis_title="", xaxis_title="Number of hardcoded strings",
    xaxis=dict(gridcolor=GRID_COLOR),
)
fig_topics.show()

callout_rows = []
for topic in topic_order:
    bucket = [s for s in strings if s.get("topic") == topic and s["length"] > 60]
    if bucket:
        pick = max(bucket, key=lambda s: s["length"])
        callout_rows.append((topic, TOPIC_COLORS.get(topic,"#8b949e"),
                             pick["file"], pick["text"][:150]))

cards_html = "".join(
    f'<div style="border-left:3px solid {color};padding:8px 12px;margin:5px 0;'
    f'background:#21262d;border-radius:0 6px 6px 0">'
    f'<span style="color:{color};font-weight:700;font-size:12px">{topic}</span>'
    f'<span style="color:#8b949e;font-size:11px;margin-left:8px">{fname}</span>'
    f'<div style="color:{FONT_COLOR};font-size:12px;margin-top:4px;'
    f'font-family:monospace;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
    f'{text}...</div></div>'
    for topic, color, fname, text in callout_rows
)
display(HTML(
    f'<div style="background:{DARK_BG};padding:16px;border-radius:10px;'
    f'border:1px solid #30363d;margin-top:10px">'
    f'<div style="color:{FONT_COLOR};font-family:Arial Black;font-size:13px;margin-bottom:10px">'
    f"Longest string per topic — Claude's actual internal directives</div>"
    f'{cards_html}</div>'
))
fig_topics.write_image("graphs/04_string_topics.png", width=1400, height=800, scale=2)

## 7 · Memory Architecture

Claude doesn't have unlimited memory. Here is the exact machinery — what gets kept,
what hits disk, and **where Claude forgets** (spoiler: `compact(keep_last=10)`).


In [12]:
memory_files = {
    "session_store.py":       "Persists full sessions (messages + tokens) to disk as JSON under .port_sessions/",
    "transcript.py":          "In-memory transcript — compact(keep_last=10) truncates old turns HERE",
    "history.py":             "HistoryLog: title + detail events per runtime session turn",
    "memdir/__init__.py":     f"'{subs.get('memdir',{}).get('archive_name','memdir')}' subsystem — "
                              f"{subs.get('memdir',{}).get('module_count','?')} archived TS modules",
    "state/__init__.py":      f"'{subs.get('state',{}).get('archive_name','state')}' subsystem — "
                              f"{subs.get('state',{}).get('module_count','?')} archived TS modules",
    "migrations/__init__.py": f"'{subs.get('migrations',{}).get('archive_name','migrations')}' subsystem — "
                              f"{subs.get('migrations',{}).get('module_count','?')} archived TS modules",
}
for fname, desc in memory_files.items():
    label = node_label(fname) if fname.endswith("__init__.py") else Path(fname).stem
    print(f"  {label:25s}  {desc}")


  session_store              Persists full sessions (messages + tokens) to disk as JSON under .port_sessions/
  transcript                 In-memory transcript — compact(keep_last=10) truncates old turns HERE
  history                    HistoryLog: title + detail events per runtime session turn
  memdir                     'memdir' subsystem — 8 archived TS modules
  state                      'state' subsystem — 6 archived TS modules
  migrations                 'migrations' subsystem — 11 archived TS modules


In [13]:
fig_mem = go.Figure(go.Sankey(
    node=dict(
        label=["User Prompt","QueryEngine","TranscriptStore",
               "SessionStore","Disk","HistoryLog","RuntimeSession"],
        color=["#58a6ff","#f78166","#56d364","#d2a8ff","#8b949e","#ffa657","#3fb950"],
        pad=22, thickness=28,
        line=dict(color="#30363d", width=0.5),
    ),
    link=dict(
        source=[0,1,1,2,3,1],
        target=[1,2,6,3,4,5],
        value= [10,7,8,5,5,6],
        label=["submit_message","append entry","bootstrap_session",
               "persist_session","write JSON","add event"],
        color=["rgba(88,166,255,0.22)"]*6,
    ),
))
fig_mem.update_layout(
    title=dict(text="🗃️ Claude Memory Data Flow", x=0.02,
               font=dict(size=17, color=FONT_COLOR, family="Arial Black, Arial")),
    height=380, paper_bgcolor=PAPER_BG,
    font=dict(family="Arial, sans-serif", size=12, color=FONT_COLOR),
    margin=dict(t=64, b=30, l=30, r=30),
)
fig_mem.show()
fig_mem.write_image("graphs/05_memory_flow.png", width=1400, height=800, scale=2)

## 8 · Commands & Tools by Subsystem

207 slash commands + 184 tools. A grouped view exposes which subsystem
owns the most and where the real command density lives.


In [14]:
cmd_counts  = Counter(hint_sub(c.get("source_hint","")) for c in cmds)
tool_counts = Counter(hint_sub(t.get("source_hint","")) for t in tools)

top_n   = 14
cmd_df  = pd.DataFrame(cmd_counts.most_common(top_n),  columns=["Subsystem","Commands"])
tool_df = pd.DataFrame(tool_counts.most_common(top_n), columns=["Subsystem","Tools"])
merged  = (
    cmd_df.merge(tool_df, on="Subsystem", how="outer")
    .fillna(0).sort_values("Commands")
)
merged["Commands"] = merged["Commands"].astype(int)
merged["Tools"]    = merged["Tools"].astype(int)

fig_ct = go.Figure()
fig_ct.add_trace(go.Bar(
    y=merged["Subsystem"], x=merged["Commands"], name="Commands",
    orientation="h", marker=dict(color="#58a6ff", line_width=0),
    text=merged["Commands"].replace(0,""), textposition="outside",
))
fig_ct.add_trace(go.Bar(
    y=merged["Subsystem"], x=merged["Tools"], name="Tools",
    orientation="h", marker=dict(color="#56d364", line_width=0),
    text=merged["Tools"].replace(0,""), textposition="outside",
))
fig_ct.update_layout(barmode="group")
_style(fig_ct, f"🤖 {len(cmds)} Commands & {len(tools)} Tools by Subsystem", height=520)
fig_ct.update_layout(
    legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center",
                bgcolor="rgba(0,0,0,0)", bordercolor=GRID_COLOR),
    xaxis_title="Count", yaxis_title="",
)
fig_ct.show()
fig_ct.write_image("graphs/06_commands_tools.png", width=1400, height=800, scale=2)

## 9 · Hidden Feature Flags

Boolean flags whose names contain keywords like `enable`, `mode`, `hook`, `mcp`, `skill`.
These are the hidden switches engineers left in the codebase.
**Green = on by default. Red = off by default.**


In [15]:
\
flags    = []
pat      = re.compile(r"(\w+)\s*[:=]\s*(True|False)")
keywords = {"mode","enable","allow","gate","flag","active","init",
            "trusted","mcp","plugin","skill","hook","simple","struct","session"}

for path, content in raw_files.items():
    for i, line in enumerate(content.splitlines(), 1):
        s = line.strip()
        if s.startswith("#"):
            continue
        for m in pat.finditer(s):
            name = m.group(1)
            if any(k in name.lower() for k in keywords):
                flags.append({"file": node_label(path), "line": i,
                              "flag": name, "default": m.group(2)})

df_flags = pd.DataFrame(flags)
print(f"Feature flags found: {len(df_flags)}")
display(df_flags.drop_duplicates("flag").reset_index(drop=True))


Feature flags found: 4


,file,line,flag,default
0,direct_modes,17,active,True
1,runtime,111,trusted,True


In [16]:
if not df_flags.empty:
    deduped = df_flags.drop_duplicates("flag").copy().reset_index(drop=True)

    fig_flags = go.Figure()
    for val, color, opacity in [("True","#56d364",0.9), ("False","#f78166",0.85)]:
        sub = deduped[deduped["default"] == val]
        if sub.empty:
            continue
        fig_flags.add_trace(go.Bar(
            y=sub["flag"], x=[1] * len(sub),
            orientation="h", name=f"Default: {val}",
            marker=dict(color=color, line_width=0, opacity=opacity),
            text=[f"{row['file']}  [{row['flag']}]" for _, row in sub.iterrows()],
            textposition="inside",
            textfont=dict(size=10, color="#0d1117"),
            hovertext=[f"<b>{row['flag']}</b><br>File: {row['file']}<br>Line: {row['line']}<br>Default: {val}"
                       for _, row in sub.iterrows()],
            hoverinfo="text",
        ))

    h = max(380, len(deduped) * 28)
    _style(fig_flags, "🚩 Every Feature Flag in Claude Code", height=h)
    fig_flags.update_layout(
        barmode="stack", showlegend=True,
        legend=dict(orientation="h", y=1.04, x=0.5, xanchor="center", bgcolor="rgba(0,0,0,0)"),
        xaxis=dict(showgrid=False, showticklabels=False, zeroline=False),
        yaxis_title="",
    )
    fig_flags.show()
fig_flags.write_image("graphs/07_feature_flags.png", width=1400, height=800, scale=2)

## 10 · The 7-Stage Bootstrap Pipeline

Every time you run `claude`, this sequence fires before the first token is generated.
Any failure in an early stage aborts the pipeline entirely.


In [17]:
STAGES = [
    "User Input","Prefetch & Side Effects","Warning & Env Guards",
    "CLI Parse + Trust Gate","Setup + Agent Load","Deferred Init",
    "Mode Router","Query Engine Loop","Response Stream",
]
STAGE_COLORS = [
    "#58a6ff","#8b949e","#f78166","#ffa657",
    "#56d364","#d2a8ff","#3fb950","#e3b341","#58a6ff",
]
EDGE_LABELS = [
    "raw keystrokes","async background tasks","validate env + safety",
    "parse args, verify trust","load plugins + cmds","lazy-import subsystems",
    "route to agent or cmd","stream model tokens",
]
fig_pipe = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=STAGES, color=STAGE_COLORS, pad=26, thickness=30,
              line=dict(color="#30363d", width=0.5)),
    link=dict(
        source=list(range(len(STAGES)-1)),
        target=list(range(1,len(STAGES))),
        value=[100,98,95,92,90,88,85,85],
        label=EDGE_LABELS,
        color=["rgba(88,166,255,0.22)"]*(len(STAGES)-1),
    ),
))
fig_pipe.update_layout(
    title=dict(text="🔄 Claude Code's 7-Stage Bootstrap -> Query Pipeline", x=0.02,
               font=dict(size=17, color=FONT_COLOR, family="Arial Black, Arial")),
    height=420, paper_bgcolor=PAPER_BG,
    font=dict(family="Arial, sans-serif", size=12, color=FONT_COLOR),
    margin=dict(t=64, b=30, l=30, r=80),
)
fig_pipe.show()
fig_pipe.write_image("graphs/08_bootstrap_pipeline.png", width=1400, height=800, scale=2)

## 11 · The Longest Import Dependency Chain

After removing cycles, the DAG longest path shows the deepest chain of transitive
dependencies — a proxy for architectural coupling. The **orange arrow** marks the
most significant PageRank transition in the chain.


In [18]:
dag = G.copy()
for _ in range(500):
    try:
        cycle = nx.find_cycle(dag)
        dag.remove_edge(*cycle[0])
    except nx.NetworkXNoCycle:
        break

try:
    chain = nx.dag_longest_path(dag)
except Exception:
    chain = []

print(f"Chain length : {len(chain)} files")
print(" -> ".join(node_label(n) for n in chain))


Chain length : 5 files
main -> runtime -> system_init -> setup -> deferred_init


In [19]:
if chain:
    n      = len(chain)
    labels = [node_label(c) for c in chain]
    colors = [sub_color.get(node_sub(c), "#8b949e") for c in chain]
    sizes  = [24 + pr.get(c, 0) * 4000 for c in chain]

    pr_vals = [pr.get(c, 0) for c in chain]
    drops   = [abs(pr_vals[i+1] - pr_vals[i]) for i in range(n-1)]
    key_i   = drops.index(max(drops)) if drops else 0

    fig_chain = go.Figure()

    for i in range(n - 1):
        is_key  = (i == key_i)
        fig_chain.add_annotation(
            ax=i, ay=0, x=i+1, y=0,
            xref="x", yref="y", axref="x", ayref="y",
            showarrow=True, arrowhead=2, arrowsize=1.3,
            arrowwidth=3 if is_key else 1.5,
            arrowcolor="#ffa657" if is_key else "#58a6ff",
        )

    fig_chain.add_trace(go.Scatter(
        x=list(range(n)), y=[0]*n,
        mode="markers+text",
        marker=dict(size=sizes, color=colors,
                    line=dict(width=2, color=PAPER_BG)),
        text=labels,
        textposition="top center",
        textfont=dict(size=11, color=FONT_COLOR),
        hovertext=[
            f"<b>{node_label(c)}</b><br>PageRank: {pr.get(c,0):.4f}<br>Subsystem: {node_sub(c)}"
            for c in chain
        ],
        hoverinfo="text",
    ))

    if n > 1:
        fig_chain.add_annotation(
            x=(key_i + key_i + 1) / 2, y=-0.42,
            text=f"key handoff: {labels[key_i]} -> {labels[key_i+1]}",
            showarrow=False,
            font=dict(size=10, color="#ffa657"),
            bgcolor="rgba(255,166,87,0.10)",
            bordercolor="#ffa657", borderwidth=1, borderpad=5,
        )

    fig_chain.add_annotation(
        x=(n-1)/2, y=0.82,
        text=f"Chain depth: {n} files | {n-1} transitive imports",
        showarrow=False,
        font=dict(size=11, color=FONT_COLOR),
        bgcolor=PAPER_BG, bordercolor=GRID_COLOR, borderwidth=1, borderpad=6,
    )

    fig_chain.update_layout(
        title=dict(text="🔗 Longest Import Dependency Chain", x=0.02,
                   font=dict(size=17, color=FONT_COLOR, family="Arial Black, Arial")),
        height=320,
        paper_bgcolor=PAPER_BG, plot_bgcolor=DARK_BG,
        font=dict(family="Arial, sans-serif", color=FONT_COLOR),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.5, n-0.5]),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-0.9, 1.3]),
        margin=dict(t=64, b=20, l=20, r=20),
    )
    fig_chain.show()
fig_chain.write_image("graphs/09_dependency_chain.png", width=1400, height=800, scale=2)

## 13 · Subsystem Architecture

Each subsystem is a compiled archive of TypeScript modules.
Module count reveals where Claude's complexity actually lives.


In [20]:
sub_rows = [
    {"Subsystem": name, "Modules": int(data.get("module_count",0)),
     "Archive": data.get("archive_name", name)}
    for name, data in subs.items() if data.get("module_count",0) > 0
]
df_heat = pd.DataFrame(sub_rows).sort_values("Modules", ascending=False)

fig_heat = px.treemap(
    df_heat, path=["Subsystem"], values="Modules",
    color="Modules",
    color_continuous_scale=[[0,"#0d2818"],[0.4,"#1a5c35"],[1,"#56d364"]],
    hover_data={"Archive": True, "Modules": True},
    custom_data=["Archive","Modules"],
)
fig_heat.update_traces(
    texttemplate="<b>%{label}</b><br>%{value} modules",
    textfont=dict(family="Arial, sans-serif", size=13),
    marker_line_width=3, marker_line_color=DARK_BG,
    hovertemplate="<b>%{label}</b><br>Archive: %{customdata[0]}<br>"
                  "Modules: %{customdata[1]}<extra></extra>",
)
_style(fig_heat, "🏗️ Subsystem Architecture — TypeScript module count per subsystem", height=560)
fig_heat.update_layout(coloraxis_colorbar=dict(
    title="Modules",
    tickfont=dict(color=FONT_COLOR),
    title_font=dict(color=FONT_COLOR),
))
fig_heat.show()
fig_heat.write_image("graphs/11_subsystem_architecture.png", width=1400, height=800, scale=2)